# 成功したプレビューを診断する

Query Doctor は保存済みの成功 SELECT を対象に、現在の計画・統計・保持履歴を限定的に読み、ローカルモデルを1回呼びます。元の SELECT は再実行しません。COUNT は対象外です。根本原因や改善効果を確定する診断ではありません。

初回 Run All は実行フラグが False なので、診断・Reader・モデルを呼びません。


In [ ]:
from IPython.display import HTML, display
from tools.local_experiment.notebook_lab.query import (
    diagnose, preview_read_token, recover_diagnose, render_diagnose,
)
from tools.local_experiment.notebook.client import NotebookError


## 診断対象を選ぶ

成功した Lab プレビューの `attempt_token` を `preview_attempt_token` に指定します。対応する Reader token は保存証跡から自動で取得します。既に SELECT の Reader token が分かっている場合は、代わりに `read_token` だけを指定できます。Lab・Assist・COUNT の token を Reader token として指定しないでください。

2つの token 欄の片方を指定し、`run_diagnose = True` にしてから次のセルを実行します。保存された診断を読むだけなら、最後の再表示セルを使います。


In [ ]:
preview_attempt_token = ""
read_token = ""
run_diagnose = False
diagnose_result = None
diagnose_html = None


In [ ]:
diagnose_result = None
diagnose_html = None
candidate = None
candidate_html = None
source_read_token = None
if run_diagnose:
    if bool(preview_attempt_token) == bool(read_token):
        print("成功プレビューの attempt_token または SELECT の Reader token の片方を指定してください。")
    else:
        try:
            source_read_token = preview_read_token(preview_attempt_token) if preview_attempt_token else read_token
            candidate = diagnose(source_read_token)
            candidate_html = render_diagnose(candidate)
        except NotebookError as error:
            candidate = None
            candidate_html = None
            print("診断は完了しませんでした。保存証跡を確認してください:", str(error))
        except BaseException:
            candidate = None
            candidate_html = None
            raise
        else:
            diagnose_result = candidate
            diagnose_html = candidate_html
            display(HTML(diagnose_html))


## 保存結果を再表示する

成功した診断の Lab `attempt_token` を指定します。保存された診断と監査証跡だけを検証し、追加の SQL・Reader・モデル呼出は行いません。再表示は当時の結果です。


In [ ]:
diagnose_attempt = ""
saved_diagnosis = None
saved_diagnosis_html = None
saved_candidate = None
saved_candidate_html = None
if diagnose_attempt:
    try:
        saved_candidate = recover_diagnose(diagnose_attempt)
        saved_candidate_html = render_diagnose(saved_candidate)
    except NotebookError as error:
        saved_candidate = None
        saved_candidate_html = None
        print("保存された診断は確認できませんでした:", str(error))
    except BaseException:
        saved_candidate = None
        saved_candidate_html = None
        raise
    else:
        saved_diagnosis = saved_candidate
        saved_diagnosis_html = saved_candidate_html
        display(HTML(saved_diagnosis_html))
